## Purpose:
    This script generates attribute conditioned Markov baseline routes using traveler-level attributes and observed graph-connected GPS routes.

## Input:
    - Hexagon road network graph:
      ./data/new_hexagraph/hexa_network_with_road.gpickle
    - Training traveler attribute file:
      ./data/sample_processed_inputs/Training/properties_for_traveler.csv
    - Training route file:
      ./data/sample_processed_inputs/Training/shortcut_route/gps_shortcut.csv
    - Validation traveler attribute file:
      ./data/sample_processed_inputs/Validation/properties_for_traveler.csv
    - Validation route file:
      ./data/sample_processed_inputs/Validation/shortcut_route/route_split_{k}/
      where k = 8, 12, and 16

## Output:
    - Segment-level personalized Markov route predictions:
      ./expected_outputs/code6.5_prediction_markov_2/markov2_route_split_{k}.csv

    - Traveler-level personalized Markov route predictions:
      ./expected_outputs/code6.5_prediction_markov_2/markov2_route_{k}.csv

## Main procedures:
    1. Load the hexagon road network graph.
    2. Construct the directed adjacency structure from the road network graph.
    3. Restore selected categorical traveler attributes from one-hot encoded columns.
    4. Estimate attribute-specific transition probabilities from training routes.
    5. Combine transition probabilities according to each validation traveler's attributes.
    6. Convert combined transition probabilities into edge costs using -log(p).
    7. Generate the most probable path for each validation OD segment using Dijkstra's algorithm.
    8. Concatenate segment-level predictions into traveler-level predicted routes.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import pickle
from torch_geometric.utils import from_networkx
import torch
import ast
import re
from collections import defaultdict
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
# Load the hexagon road network graph.
with open("./data/new_hexagraph/hexa_network_with_road.gpickle", 'rb') as f:
    G = pickle.load(f)
data = from_networkx(G)

num_nodes = data.num_nodes
data.x = torch.eye(num_nodes)
node_feat_dim = data.x.size(1)

node_features = data.x
edge_index = data.edge_index

out_neighbors = {u: [] for u in range(num_nodes)}
edge_allowed = set()

for i in range(edge_index.shape[1]):
    u = int(edge_index[0, i].item())  # 0..N-1
    v = int(edge_index[1, i].item())  # 0..N-1
    out_neighbors[u].append(v)
    edge_allowed.add((u, v))

## Training

### preprocess_traveler_attribute_dataframes

In [5]:
# Step 1: Load traveler attributes and training shortcut routes.
df_tr     = pd.read_csv("./data/sample_processed_inputs/Training/properties_for_traveler.csv")       # attribute data for each traveler
route_df  = pd.read_csv("./data/sample_processed_inputs/Training/shortcut_route/gps_shortcut.csv")        # 'name', 'route' columns

# Step 2: Convert route strings into Python lists and extract traveler IDs.
route_df['route'] = route_df['path'].apply(ast.literal_eval)
route_df['TRAVELER_ID'] = route_df['name'].str[2:-2]

# Step 3: Restore selected categorical attributes from one-hot encoded columns.
purpose_cols = [f"travel_purpose_{i}" for i in range(1,51)]
df_tr["TRAVEL_PURPOSE"] = (
    df_tr[purpose_cols]
      .idxmax(axis=1)
      .str.extract(r'(\d+)')
      .astype(int)
)

for i in range(1,9):
    style_cols = [f"TRAVEL_STYL_{i}_{j}" for j in range(1,8)]
    df_tr[f"TRAVEL_STYLE_{i}"] = (
        df_tr[style_cols]
          .idxmax(axis=1)
          .str.extract(r'TRAVEL_STYL_\d+_(\d+)')
          .astype(int)
    )

move_cols = [f"MOST_MVMN_CD_1_{i}" for i in range(1,17)]
df_tr["MOST_MVMN_CD"] = (
    df_tr[move_cols]
      .idxmax(axis=1)
      .str.extract(r'(\d+)')
      .astype(int)
)

# Step 4: Define traveler attributes used for personalized transition modeling.
selected = [
    "GENDER", "AGE_GRP", "TRAVEL_COMPANIONS_NUM",
    "TRAVEL_TERM", "TRAVEL_PURPOSE"
] + [f"TRAVEL_STYLE_{i}" for i in range(1,9)] + ["MOST_MVMN_CD"]

# Step 5: Merge training routes with traveler attributes.
df = pd.merge(
    route_df[['TRAVELER_ID','route']],
    df_tr[['TRAVELER_ID'] + selected],
    on='TRAVELER_ID', how='inner'
)

### estimating attribute-specific transition probabilities

In [6]:
# =========================
# Define attribute specifications
# =========================
ATTR_SPECS = {
    "GENDER": [0, 1],
    "AGE_GRP": [10,20,30,40,50,60,70],
    "TRAVEL_COMPANIONS_NUM": [0,1,2,3,4,5,6,7,8],
    "TRAVEL_TERM": [1,2,3,4],
    "TRAVEL_PURPOSE": list(range(1,51)),
    # Eight travel-style dimensions.
    **{f"TRAVEL_STYLE_{i}": list(range(1,8)) for i in range(1,9)},
    "MOST_MVMN_CD": list(range(1,17))
}

# count traveler routes by attribute level and edge (with smoothing)
def accumulate_counts_by_attribute(df, attr_name, levels, epsilon=1e-10):
    """
      counts[level][(u,v)] = count + epsilon
      row_sums[level][u]   = sum of counts for all outgoing edges from u at this level
    """
    counts   = {lvl: defaultdict(float) for lvl in levels}
    row_sums = {lvl: defaultdict(float) for lvl in levels}

    # Add epsilon smoothing to all allowed outgoing edges.
    for lvl in levels:
        for u, nbrs in out_neighbors.items():
            if not nbrs:
                continue
            base = epsilon * len(nbrs)
            row_sums[lvl][u] += base
            for v in nbrs:
                counts[lvl][(u, v)] += epsilon

    # Accumulate observed transitions from training routes.
    for _, row in df.iterrows():
        lvl = int(row[attr_name])
        if lvl not in levels:
            continue
        r = row['route']
        if not isinstance(r, (list, tuple)) or len(r) < 2:
            continue
        for i in range(len(r) - 1):
            s = int(r[i]); e = int(r[i+1])
            if (s, e) in edge_allowed:
                counts[lvl][(s, e)] += 1.0
                row_sums[lvl][s]   += 1.0

    return counts, row_sums

# Create caches for count/row sum by attribute
attr_counts   = {}
attr_row_sums = {}
for attr, levels in ATTR_SPECS.items():
    cts, sums = accumulate_counts_by_attribute(df, attr, levels, epsilon=1e-10)
    attr_counts[attr]   = cts
    attr_row_sums[attr] = sums

# p_attr(level | u->v)
def prob_attr_level(attr, level, u, v, tiny=1e-15):
    cts  = attr_counts[attr][level]
    rsum = attr_row_sums[attr][level].get(u, 0.0)
    if rsum <= 0:
        return 0.0
    return float(cts.get((u, v), 0.0)) / max(rsum, tiny)

# Join attribute levels → combined transition probability
def combined_transition_prob_sparse(target_attrs, attr_weights=None, tiny=1e-15):
    use_attrs = []
    for attr, level in target_attrs.items():
        if (attr in ATTR_SPECS) and (level in ATTR_SPECS[attr]):
            use_attrs.append((attr, level))
    if not use_attrs:
        # If no valid attributes are matched, use a uniform distribution over allowed edges.
        def uniform_prob(u, v):
            deg = len(out_neighbors[u])
            return (1.0 / deg) if (deg > 0 and v in out_neighbors[u]) else 0.0
        return uniform_prob

    if attr_weights is None:
        attr_weights = {}

    def get_prob(u, v):
        if (u, v) not in edge_allowed:
            return 0.0
        prod = 1.0
        for (attr, level) in use_attrs:
            w = float(attr_weights.get(attr, 1.0))
            p = prob_attr_level(attr, level, u, v, tiny=tiny)
            if p <= 0.0:
                return 0.0
            prod *= (p ** w) if w != 1.0 else p
        return prod

    return get_prob

# Convert a row of combined transition probabilities into a distribution over outgoing edges.
def combined_row_distribution(u, get_prob):
    nbrs = out_neighbors[u]
    if not nbrs:
        return [], np.array([])
    vals = np.array([get_prob(u, v) for v in nbrs], dtype=float)
    s = vals.sum()
    if s <= 0:
        # If all combined probabilities are zero, use a uniform distribution over allowed outgoing edges.
        vals = np.ones_like(vals) / len(nbrs)
    else:
        vals = vals / s
    return nbrs, vals

# Build a graph with edge weights derived from combined transition probabilities.
def build_graph_from_combined(get_prob, tiny=1e-15):
    Gx = nx.DiGraph()
    for u, nbrs in out_neighbors.items():
        if not nbrs:
            continue
        # Convert row-normalized transition probabilities into edge costs using -log(p).
        vs, probs = combined_row_distribution(u, get_prob)
        for v, p in zip(vs, probs):
            p = max(float(p), tiny)
            Gx.add_edge(u, v, weight=-np.log(p))
    return Gx

def make_route(Gx, source_0, target_0):
    try:
        return nx.dijkstra_path(Gx, source=source_0, target=target_0, weight='weight')
    except nx.NetworkXNoPath:
        return None

## Validation

In [7]:
def numerical_sort_key(file):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', file)]

def flatten_ext(routes):
    return [p for route in routes for p in route]

In [8]:
# 1. Load input data
df_va     = pd.read_csv("./data/sample_processed_inputs/Validation/properties_for_traveler.csv")
route_df_va  = pd.read_csv("./data/sample_processed_inputs/Validation/shortcut_route/gps_shortcut.csv")

# 2. Convert route strings to lists and extract traveler IDs
route_df_va['route'] = route_df_va['path'].apply(ast.literal_eval)
route_df_va['TRAVELER_ID'] = route_df_va['name'].str[2:-2]  # Extract traveler ID

# 3. Restore selected attributes from one-hot encoded columns
purpose_cols = [f"travel_purpose_{i}" for i in range(1,51)]
df_va["TRAVEL_PURPOSE"] = (
    df_va[purpose_cols]
      .idxmax(axis=1)
      .str.extract(r'(\d+)')
      .astype(int)
)

for i in range(1,9):
    style_cols = [f"TRAVEL_STYL_{i}_{j}" for j in range(1,8)]
    df_va[f"TRAVEL_STYLE_{i}"] = (
        df_va[style_cols]
          .idxmax(axis=1)
          .str.extract(r'TRAVEL_STYL_\d+_(\d+)')
          .astype(int)
    )

move_cols = [f"MOST_MVMN_CD_1_{i}" for i in range(1,17)]
df_va["MOST_MVMN_CD"] = (
    df_va[move_cols]
      .idxmax(axis=1)
      .str.extract(r'(\d+)')
      .astype(int)
)

# 4. Define attributes used for personalized Markov modeling
selected = [
    "GENDER", "AGE_GRP", "TRAVEL_COMPANIONS_NUM",
    "TRAVEL_TERM", "TRAVEL_PURPOSE"
] + [f"TRAVEL_STYLE_{i}" for i in range(1,9)] + ["MOST_MVMN_CD"]

# 5. Merge route records with traveler attributes
df_va = pd.merge(
    route_df_va[['TRAVELER_ID']],
    df_va[['TRAVELER_ID'] + selected],
    on='TRAVELER_ID', how='inner'
)

In [9]:
# Generate personalized Markov route predictions for each OD segment.
for k in range(8, 17, 4):
    # Load validation route segment files for the current window size.
    csv_dir_Va = f'./data/sample_processed_inputs/Validation/shortcut_route/route_split_{k}'
    route_split_list_Va = sorted(
        [f for f in os.listdir(csv_dir_Va) if f.endswith(".csv")],
        key=numerical_sort_key
    )

    results = []
    for file_name in route_split_list_Va:
        tmp_r = pd.read_csv(os.path.join(csv_dir_Va, file_name))
        travel_name = tmp_r.iloc[0,1]
        df_tmp = df_va[df_va["TRAVELER_ID"] == tmp_r.iloc[0,1][:-2]]
        target_attrs = {
            "GENDER": df_tmp.iloc[0,1],
            "AGE_GRP": df_tmp.iloc[0,2],
            "TRAVEL_COMPANIONS_NUM": df_tmp.iloc[0,3],
            "TRAVEL_TERM": df_tmp.iloc[0,4],
            "TRAVEL_PURPOSE": df_tmp.iloc[0,5],
            "TRAVEL_STYLE_1": df_tmp.iloc[0,6],
            "TRAVEL_STYLE_2": df_tmp.iloc[0,7],
            "TRAVEL_STYLE_3": df_tmp.iloc[0,8],
            "TRAVEL_STYLE_4": df_tmp.iloc[0,9],
            "TRAVEL_STYLE_5": df_tmp.iloc[0,10],
            "TRAVEL_STYLE_6": df_tmp.iloc[0,11],
            "TRAVEL_STYLE_7": df_tmp.iloc[0,12],
            "TRAVEL_STYLE_8": df_tmp.iloc[0,13],
            "MOST_MVMN_CD": df_tmp.iloc[0,14]
        }
        # Combine attribute-specific transition probabilities for the target traveler.
        get_prob = combined_transition_prob_sparse(target_attrs)
        # Build a weighted directed graph using -log(p) as edge costs.
        G_combined = build_graph_from_combined(get_prob)

        start_idx = int(tmp_r.iloc[0, 0])
        end_idx   = int(tmp_r.iloc[-1, 0])
        travel_name = tmp_r.iloc[0, 1] if tmp_r.shape[1] > 1 else f"travel_{file_name}"
        
        # Generate the maximum-probability route using Dijkstra's algorithm.
        route_pred = make_route(G_combined, start_idx, end_idx)
        results.append({'travel_name': travel_name, 'predicted_route': route_pred or []})

    out_dir = './expected_outputs/code6.5_prediction_markov_2'
    os.makedirs(out_dir, exist_ok=True)
    df_result = pd.DataFrame(results)
    df_result.to_csv(f'{out_dir}/markov2_route_split_{k}.csv', index=False)

    grouped_df = df_result.groupby('travel_name')['predicted_route'].apply(flatten_ext).reset_index()
    grouped_df.to_csv(f'{out_dir}/markov2_route_{k}.csv', index=False)
    print("finish", k)

print("All done (method-2: all attributes × elementwise, sparse).")

finish 8
finish 12
finish 16
All done (method-2: all attributes × elementwise, sparse).
